# GeoLibre 2.0: GPX, KML y GeoJSON con animación de ruta

Notebook reproducible para cargar una ruta real aportada por el usuario, normalizarla a GeoJSON y abrirla en GeoLibre. La animación del marcador se activa en la interfaz con **Controls → Route Animation**. Una línea importada no es una ruta de evacuación validada.

In [ ]:
%pip install -q "geolibre[all]==2.0.0" geopandas pyogrio shapely

## Formatos y librerías

| Formato | Contenido útil | Precaución |
|---|---|---|
| GPX | routes, tracks, waypoints, elevación y tiempo | No contiene estado oficial de carreteras |
| KML | geometría, estilos, jerarquías, tiempo y overlays | Las extensiones `gx:*`, modelos y overlays pueden perderse al convertir |
| GeoJSON | geometría web y propiedades en WGS84 | No define por sí mismo seguridad, tiempos ni semántica de evacuación |

GeoPandas/Pyogrio ofrece un flujo común. Para análisis GPX detallado conviene `gpxpy`; para KML avanzado, `fastkml`; para máxima cobertura, GDAL/OGR.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
import pyogrio
from geolibre import Map, __version__ as geolibre_version

print("GeoLibre", geolibre_version)
ROUTE_PATH = Path("ruta_real.gpx")  # Cambia por un .gpx, .kml, .geojson o .json real
if not ROUTE_PATH.exists():
    raise FileNotFoundError(f"Copia tu archivo real en {ROUTE_PATH.resolve()} o cambia ROUTE_PATH")

In [ ]:
layers = pyogrio.list_layers(ROUTE_PATH)
print("Capas detectadas:")
for layer_name, geometry_type in layers:
    print(f"- {layer_name}: {geometry_type}")

In [ ]:
def load_linear_routes(path: Path) -> gpd.GeoDataFrame:
    frames = []
    for layer_name, _geometry_type in pyogrio.list_layers(path):
        frame = gpd.read_file(path, layer=layer_name, engine="pyogrio")
        if frame.empty or frame.geometry.name not in frame.columns:
            continue
        frame = frame[frame.geometry.notna()].copy()
        frame = frame[frame.geometry.geom_type.isin(["LineString", "MultiLineString"])]
        if frame.empty:
            continue
        if frame.crs is None:
            frame = frame.set_crs(4326)
        frame = frame.to_crs(4326).explode(index_parts=False, ignore_index=True)
        frame["source_layer"] = str(layer_name)
        frames.append(frame)
    if not frames:
        raise ValueError("El archivo no contiene routes/tracks lineales legibles")
    combined = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=4326)
    west, south, east, north = combined.total_bounds
    if west < -10 or east > 5 or south < 35 or north > 44:
        raise ValueError("La ruta sale del ámbito de España configurado para METEO")
    return combined

routes = load_linear_routes(ROUTE_PATH)
routes["length_m"] = routes.to_crs(3035).length
routes = routes.sort_values("length_m", ascending=False).reset_index(drop=True)
display(routes[["source_layer", "length_m", "geometry"]].head())
print(f"{len(routes)} líneas · {routes.length_m.sum()/1000:.1f} km en total")

In [ ]:
west, south, east, north = routes.total_bounds
center = ((west + east) / 2, (south + north) / 2)
m = Map(center=center, zoom=12, basemap="positron", height="720px", layout="full")
layer_id = m.add_geojson(
    json.loads(routes.to_json()),
    name=f"Ruta de referencia · {ROUTE_PATH.name}",
    strokeColor="#2563eb",
    strokeWidth=4,
)
m

## Animación oficial de GeoLibre

1. En el widget anterior abre **Controls → Route Animation**.
2. Selecciona la capa `Ruta de referencia`.
3. Elige flecha o punto, activa el rastro y ajusta la velocidad visual.
4. Activa el seguimiento solo si no dificulta interpretar el mapa.

La velocidad controla la reproducción; no es una velocidad segura ni un tiempo estimado de evacuación. La API Python 2.0 no expone todavía métodos dedicados para accionar ese panel.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
routes.to_file(output_dir / "ruta-normalizada.geojson", driver="GeoJSON")
m.save_project(output_dir / "ruta.geolibre.json")
m.to_html(output_dir / "ruta-geolibre.html", title="Ruta de referencia")
print("Exportado en", output_dir.resolve())

## Alternativa programática: mover la cámara

`Map.fly_to()` anima la cámara, no el marcador de Route Animation. Su argumento `duration` está expresado en **milisegundos**. El siguiente bloque es opcional y recorre vértices de la línea principal sin calcular seguridad ni tiempos de llegada.

In [ ]:
import time

primary_coordinates = list(routes.iloc[0].geometry.coords)
step = max(1, len(primary_coordinates) // 200)
for longitude, latitude, *_ in primary_coordinates[::step]:
    m.fly_to(longitude, latitude, zoom=14, duration=300)
    time.sleep(0.35)